# Adaptive Ramanujan Filter Bank (Q40)
## M(q)-Ordered vs Sequential Period Detection

**Goal:** Test whether ordering Ramanujan sum filters by |M(q)| descending
detects signal periods faster (fewer filters evaluated) than sequential ordering.

**Kill criterion:**
- M(q)-ordering uses >= 90% as many filters as sequential: NO-GO
- M(q)-ordering uses < 80% as many filters as sequential: GO (>20% savings)

**Runtime:** ~1 min (Quick), ~5 min (Full)

**GPU:** Not required.

In [ ]:
# --- CELL 1: Configuration ---
MODE = 'quick'  # 'quick' -> small params, 'full' -> complete sweep

import numpy as np
import time
import json
from math import gcd
from collections import defaultdict

if MODE == 'quick':
    Q = 60           # max filter order
    N = 256          # signal length
    PERIODS = [7, 13, 31]
    SNR_LEVELS = [5, 10, 20]
    N_NOISE_TRIALS = 20
    N_RANDOM_TRIALS = 30
else:
    Q = 120
    N = 500
    PERIODS = [7, 13, 31, 97]
    SNR_LEVELS = [0, 5, 10, 20]
    N_NOISE_TRIALS = 50
    N_RANDOM_TRIALS = 100

print(f'Mode: {MODE}')
print(f'Q={Q}, N={N}, periods={PERIODS}, SNRs={SNR_LEVELS}')

In [ ]:
# --- CELL 2: Mertens function ---

def mertens_function(n_max):
    """Compute Mertens function M(n) for n=1..n_max via Mobius sieve."""
    mu = [0] * (n_max + 1)
    mu[1] = 1
    is_prime = [True] * (n_max + 1)
    primes = []
    for i in range(2, n_max + 1):
        if is_prime[i]:
            primes.append(i)
            mu[i] = -1
        for p in primes:
            if i * p > n_max:
                break
            is_prime[i * p] = False
            if i % p == 0:
                mu[i * p] = 0
                break
            else:
                mu[i * p] = -mu[i]
    M = [0] * (n_max + 1)
    for i in range(1, n_max + 1):
        M[i] = M[i - 1] + mu[i]
    return M, mu

M_func, mu = mertens_function(Q)
print(f'Mertens function computed for q=1..{Q}')
print(f'M({Q}) = {M_func[Q]}')

In [ ]:
# --- CELL 3: Ramanujan sum and filter utilities ---

def ramanujan_sum(q, n):
    """Compute c_q(n) = sum_{a=1, gcd(a,q)=1}^{q} cos(2*pi*a*n/q)."""
    total = 0.0
    for a in range(1, q + 1):
        if gcd(a, q) == 1:
            total += np.cos(2 * np.pi * a * n / q)
    return total

def precompute_ramanujan_matrix(Q_max, N_sig):
    """Precompute c_q(n) for q=1..Q_max, n=0..N_sig-1."""
    cq = {}
    for q in range(1, Q_max + 1):
        arr = np.zeros(N_sig)
        for n in range(N_sig):
            arr[n] = ramanujan_sum(q, n)
        cq[q] = arr
    return cq

def generate_signal(P, N_sig, snr_db):
    """Generate sin(2*pi*n/P) + noise at given SNR."""
    n = np.arange(N_sig)
    signal = np.sin(2 * np.pi * n / P)
    if snr_db is None or snr_db == float('inf'):
        return signal
    sig_power = np.mean(signal ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.random.randn(N_sig) * np.sqrt(noise_power)
    return signal + noise

def detect_in_order(x, cq_matrix, q_order, true_period):
    """Evaluate filters in given order. Return number needed to detect true period."""
    energies_so_far = {}
    for i, q in enumerate(q_order):
        inner = np.dot(x, cq_matrix[q])
        energies_so_far[q] = inner ** 2
        current_best = max(energies_so_far, key=energies_so_far.get)
        if current_best == true_period:
            return i + 1
    return len(q_order)

print('Utility functions defined.')

In [ ]:
# --- CELL 4: Precompute Ramanujan matrix ---
print(f'Precomputing Ramanujan sum matrix ({Q} x {N})...')
t0 = time.time()
cq_matrix = precompute_ramanujan_matrix(Q, N)
print(f'Done in {time.time()-t0:.1f}s')

# Build orderings
q_values = list(range(1, Q + 1))
seq_order = list(q_values)
mertens_order = sorted(q_values, key=lambda q: (-abs(M_func[q]), q))

print(f'\nTop-15 M(q)-ordered filters: {mertens_order[:15]}')
print(f'Their |M(q)| values: {[abs(M_func[q]) for q in mertens_order[:15]]}')

In [ ]:
# --- CELL 5: Run experiment ---
t0 = time.time()
results = []

print(f'{"Period":>6} {"SNR":>5} {"Seq":>6} {"M(q)":>6} {"Random":>8} {"M/S":>7} {"R/S":>7} {"Verdict":>10}')
print('-' * 65)

all_ratios = []

for P in PERIODS:
    for snr_db in SNR_LEVELS:
        seq_counts = []
        mert_counts = []
        rand_counts = []

        for trial in range(N_NOISE_TRIALS):
            np.random.seed(trial * 1000 + P * 100 + snr_db)
            x = generate_signal(P, N, snr_db)

            sc = detect_in_order(x, cq_matrix, seq_order, P)
            seq_counts.append(sc)

            mc = detect_in_order(x, cq_matrix, mertens_order, P)
            mert_counts.append(mc)

            trial_rand = []
            for rt in range(N_RANDOM_TRIALS):
                rng = np.random.RandomState(trial * 10000 + rt)
                rand_order = list(q_values)
                rng.shuffle(rand_order)
                rc = detect_in_order(x, cq_matrix, rand_order, P)
                trial_rand.append(rc)
            rand_counts.append(np.mean(trial_rand))

        avg_seq = np.mean(seq_counts)
        avg_mert = np.mean(mert_counts)
        avg_rand = np.mean(rand_counts)

        ratio_ms = avg_mert / avg_seq if avg_seq > 0 else 1.0
        ratio_rs = avg_rand / avg_seq if avg_seq > 0 else 1.0

        if ratio_ms < 0.80:
            verdict = 'GO'
        elif ratio_ms < 0.90:
            verdict = 'MARGINAL'
        else:
            verdict = 'NO-GO'

        all_ratios.append(ratio_ms)
        results.append({
            'period': P, 'snr': snr_db,
            'seq': float(avg_seq), 'mertens': float(avg_mert), 'random': float(avg_rand),
            'ratio_ms': float(ratio_ms), 'ratio_rs': float(ratio_rs),
            'verdict': verdict
        })

        print(f'{P:6d} {snr_db:5d} {avg_seq:6.1f} {avg_mert:6.1f} {avg_rand:8.1f} {ratio_ms:7.3f} {ratio_rs:7.3f} {verdict:>10}')

total_time = time.time() - t0
print(f'\nTotal time: {total_time:.1f}s')

In [ ]:
# --- CELL 6: Summary and verdict ---

mean_ratio = np.mean(all_ratios)
n_go = sum(1 for r in results if r['verdict'] == 'GO')
n_marginal = sum(1 for r in results if r['verdict'] == 'MARGINAL')
n_nogo = sum(1 for r in results if r['verdict'] == 'NO-GO')

print('=' * 60)
print('RAMANUJAN FILTER BANK - FINAL VERDICT')
print('=' * 60)
print(f'Mean M(q)/Sequential ratio: {mean_ratio:.3f}')
print(f'GO: {n_go}, MARGINAL: {n_marginal}, NO-GO: {n_nogo}')
print()

if mean_ratio < 0.80:
    overall = 'GO -- M(q)-ordering provides >20% savings on average'
elif mean_ratio < 0.90:
    overall = 'MARGINAL -- M(q)-ordering shows modest improvement'
else:
    overall = 'NO-GO -- M(q)-ordering does not meaningfully improve over sequential'

print(f'OVERALL: {overall}')

In [ ]:
# --- CELL 7: Visualization ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Bar chart: filters needed by ordering for each (period, snr)
ax = axes[0]
labels = [f'P={r["period"]},\nSNR={r["snr"]}' for r in results]
x_pos = np.arange(len(results))
w = 0.25
ax.bar(x_pos - w, [r['seq'] for r in results], w, label='Sequential', alpha=0.8)
ax.bar(x_pos, [r['mertens'] for r in results], w, label='M(q)-ordered', alpha=0.8)
ax.bar(x_pos + w, [r['random'] for r in results], w, label='Random', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=7, rotation=45)
ax.set_ylabel('Filters to detect')
ax.set_title('Filters Needed by Ordering Strategy')
ax.legend()

# 2. Ratio heatmap
ax = axes[1]
ratios_for_plot = [r['ratio_ms'] for r in results]
colors = ['green' if r < 0.8 else 'orange' if r < 0.9 else 'red' for r in ratios_for_plot]
ax.bar(x_pos, ratios_for_plot, color=colors, alpha=0.8)
ax.axhline(y=0.8, color='green', linestyle='--', label='GO threshold (0.8)')
ax.axhline(y=0.9, color='red', linestyle='--', label='NO-GO threshold (0.9)')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=7, rotation=45)
ax.set_ylabel('M(q) / Sequential ratio')
ax.set_title('M(q)-Ordering Efficiency Ratio')
ax.legend()

plt.tight_layout()
plt.savefig('ramanujan_filter_results.png', dpi=150)
plt.show()
print('Plot saved to ramanujan_filter_results.png')

In [ ]:
# --- CELL 8: Save results ---
output = {
    'mode': MODE,
    'parameters': {'Q': Q, 'N': N, 'periods': PERIODS, 'snr_levels': SNR_LEVELS},
    'mean_ratio': float(mean_ratio),
    'overall_verdict': overall,
    'go_count': n_go,
    'marginal_count': n_marginal,
    'nogo_count': n_nogo,
    'detailed_results': results,
}

with open('ramanujan_filter_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Results saved to ramanujan_filter_results.json')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil, os
    dest = '/content/drive/MyDrive/Farey_Results/'
    os.makedirs(dest, exist_ok=True)
    shutil.copy('ramanujan_filter_results.json', dest)
    shutil.copy('ramanujan_filter_results.png', dest)
    print(f'Copied to Google Drive: {dest}')
except:
    print('Google Drive not mounted. Results saved locally.')